# Pump Performance from Recirculation Logs

This notebook compares delivered HFE volume flow against pump speed using the recirculation logs in `data/raw/recirculation`.

Key choices:

- Pump inlet temperature is `TMI_C`, matching the web UI label for the pump inlet thermocouple.
- Legacy pre-April-20 runs are normalized through `orca.logbook.add_canonical_flow_columns`, which applies the proxy thermocouple reconstruction for the earlier wrong-type TC logs.
- Volume flow is `volume_flow_lmin_si`, produced by the same flow-log normalization helper.
- Error bands show mean ± one sample standard deviation within each pump-speed and inlet-temperature bin.


In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings(
    'ignore',
    message='FigureCanvasAgg is non-interactive, and thus cannot be shown',
)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 160,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'raw' / 'recirculation').exists() and (candidate / 'analysis' / 'src').exists():
            return candidate
    raise RuntimeError('Could not find repo root from current working directory.')


REPO_ROOT = find_repo_root()
ANALYSIS_SRC = REPO_ROOT / 'analysis' / 'src'
if str(ANALYSIS_SRC) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_SRC))

from orca.logbook import add_canonical_flow_columns, is_legacy_wrong_type_log, read_tc_calibrated_csv

RECIRCULATION_DIR = REPO_ROOT / 'data' / 'raw' / 'recirculation'
TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'
LOG_GLOB = 'log_*.csv'

# Broad enough to retain cold liquid HFE recirculation data while excluding empty/gas-rich readings.
PUMP_DENSITY_BOUNDS_KG_M3 = (1100.0, 1800.0)

TEMP_BIN_WIDTH_C = 5.0
MIN_SAMPLES_PER_SPEED_TEMP = 10
MIN_SPEEDS_PER_TEMP_BIN = 2
MIN_SETTLE_SECONDS = 2.0
RPM_PER_HZ_FALLBACK = 30.0

print(f'Repo root: {REPO_ROOT}')
print(f'Recirculation logs: {RECIRCULATION_DIR}')
print(f'TC calibration: {TC_CALIBRATION_PATH}')


Repo root: /home/aamy/Documents/hfe-system
Recirculation logs: /home/aamy/Documents/hfe-system/data/raw/recirculation
TC calibration: /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv


## Data Preparation

The helper below loads each recirculation CSV, applies the shared flow-log normalization, computes pump speed in rpm, and drops the beginning of each command step before binning. The wider density bounds are intentional: the cold April 24 run reaches densities above the default review range but is still useful for pump-performance comparison.


In [2]:
def finite_numeric(frame: pd.DataFrame, column: str, default: float = np.nan) -> pd.Series:
    if column in frame.columns:
        return pd.to_numeric(frame[column], errors='coerce')
    return pd.Series(default, index=frame.index, dtype='float64')


def add_pump_speed_columns(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    freq_hz = finite_numeric(data, 'pump_freq_hz').astype(float)
    speed_rpm = finite_numeric(data, 'pump_rotation_speed_rpm').astype(float)
    fallback_rpm = (freq_hz * RPM_PER_HZ_FALLBACK).to_numpy(dtype=float)

    if 'pump_rotation_speed_rpm' in data.columns:
        speed_values = speed_rpm.to_numpy(dtype=float)
        use_fallback = ~np.isfinite(speed_values) | (speed_values <= 0.0)
        data['pump_speed_rpm'] = np.where(use_fallback, fallback_rpm, speed_values)
        data['pump_speed_source'] = np.where(use_fallback, 'freq_hz_x30', 'logged_rpm')
    else:
        data['pump_speed_rpm'] = fallback_rpm
        data['pump_speed_source'] = 'freq_hz_x30'

    return data


def settled_positive_flow_samples(data: pd.DataFrame, *, log_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    work = data.copy().sort_values('time_s')
    work['cmd_bucket_pct'] = finite_numeric(work, 'pump_cmd_pct').round(0)

    fluid_meter_valid = finite_numeric(work, 'fluid_meter_valid', default=1.0).fillna(1.0).gt(0.0)
    base_mask = (
        work['pump_running'].fillna(False)
        & work['liquid_like_density'].fillna(False)
        & fluid_meter_valid
        & finite_numeric(work, 'positive_mass_flow').fillna(False).astype(bool)
        & finite_numeric(work, 'volume_flow_lmin_si').gt(0.0)
        & finite_numeric(work, 'pump_speed_rpm').gt(0.0)
        & finite_numeric(work, 'cmd_bucket_pct').gt(0.0)
        & finite_numeric(work, 'TMI_C').notna()
    )
    work = work.loc[base_mask].copy()
    if work.empty:
        return work, pd.DataFrame()

    dt_s = finite_numeric(work, 'time_s').diff().clip(lower=0.0)
    positive_dt = dt_s[dt_s.gt(0.0) & dt_s.notna()]
    median_dt_s = float(positive_dt.median()) if not positive_dt.empty else MIN_SETTLE_SECONDS
    settle_cutoff_s = max(MIN_SETTLE_SECONDS, median_dt_s)
    time_gap_break_s = max(10.0, 5.0 * median_dt_s)

    command_changed = work['cmd_bucket_pct'].ne(work['cmd_bucket_pct'].shift())
    time_gap = dt_s.fillna(0.0).gt(time_gap_break_s)
    work['step_id'] = (command_changed | time_gap).cumsum()
    work['time_from_step_s'] = work.groupby('step_id')['time_s'].transform(lambda series: series - series.iloc[0])
    work['settle_cutoff_s'] = settle_cutoff_s

    step_windows = (
        work.groupby('step_id')
        .agg(
            log_name=('log_name', 'first'),
            cmd_pct=('cmd_bucket_pct', 'first'),
            start_s=('time_s', 'min'),
            end_s=('time_s', 'max'),
            duration_s=('time_s', lambda series: float(series.iloc[-1] - series.iloc[0])),
            median_speed_rpm=('pump_speed_rpm', 'median'),
            median_inlet_temp_C=('TMI_C', 'median'),
            sample_count=('time_s', 'size'),
        )
        .reset_index()
    )
    step_windows['settle_cutoff_s'] = settle_cutoff_s

    settled = work.loc[work['time_from_step_s'].ge(settle_cutoff_s)].copy()
    return settled, step_windows


def load_one_log(path: Path) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    raw = read_tc_calibrated_csv(path, calibration_path=TC_CALIBRATION_PATH)
    data, flow_note = add_canonical_flow_columns(
        raw,
        density_bounds=PUMP_DENSITY_BOUNDS_KG_M3,
        log_path=path,
    )
    data = add_pump_speed_columns(data)
    data['log_name'] = path.name
    data['legacy_tc_correction_applied'] = bool(data.attrs.get('legacy_tc_correction_applied', False))
    data['legacy_tc_room_anchor_samples'] = int(data.attrs.get('legacy_tc_room_anchor_samples', 0) or 0)
    data['legacy_tc_effective_cold_junction_c'] = data.attrs.get('legacy_tc_effective_cold_junction_c', np.nan)
    data['restored_tc_calibration_applied'] = bool(data.attrs.get('tc_calibration_applied_from_log_metadata', False))

    settled, step_windows = settled_positive_flow_samples(data, log_name=path.name)

    coverage = pd.Series({
        'log_name': path.name,
        'raw_rows': len(raw),
        'normalized_rows': len(data),
        'settled_samples': len(settled),
        'legacy_pre_fix_log': is_legacy_wrong_type_log(path),
        'legacy_tc_correction_applied': bool(data.attrs.get('legacy_tc_correction_applied', False)),
        'restored_tc_calibration_applied': bool(data.attrs.get('tc_calibration_applied_from_log_metadata', False)),
        'legacy_room_anchor_samples': int(data.attrs.get('legacy_tc_room_anchor_samples', 0) or 0),
        'pump_speed_sources': ', '.join(sorted(set(data['pump_speed_source'].dropna().astype(str)))) or 'none',
        'tmi_min_C': float(settled['TMI_C'].min()) if not settled.empty else np.nan,
        'tmi_max_C': float(settled['TMI_C'].max()) if not settled.empty else np.nan,
        'flow_min_L_min': float(settled['volume_flow_lmin_si'].min()) if not settled.empty else np.nan,
        'flow_max_L_min': float(settled['volume_flow_lmin_si'].max()) if not settled.empty else np.nan,
        'speed_min_rpm': float(settled['pump_speed_rpm'].min()) if not settled.empty else np.nan,
        'speed_max_rpm': float(settled['pump_speed_rpm'].max()) if not settled.empty else np.nan,
        'command_levels_pct': ', '.join(str(int(v)) for v in sorted(set(settled['cmd_bucket_pct'].dropna()))) if not settled.empty else '',
        'normalization_note': flow_note,
    })
    return settled, coverage, step_windows


log_paths = sorted(path for path in RECIRCULATION_DIR.glob(LOG_GLOB) if '_pump_log_backups' not in path.parts)
print(f'Found {len(log_paths)} recirculation CSV files.')
for path in log_paths:
    print(f'  {path.name}')


Found 7 recirculation CSV files.
  log_20260330_161922.csv
  log_20260402_150754.csv
  log_20260403_115916.csv
  log_20260417_094053.csv
  log_20260422_102446_filling.csv
  log_20260422_143345.csv
  log_20260424_153546.csv


In [3]:
settled_frames: list[pd.DataFrame] = []
coverage_rows: list[pd.Series] = []
step_window_frames: list[pd.DataFrame] = []

for path in log_paths:
    settled, coverage, step_windows = load_one_log(path)
    settled_frames.append(settled)
    coverage_rows.append(coverage)
    if not step_windows.empty:
        step_window_frames.append(step_windows)

coverage = pd.DataFrame(coverage_rows).set_index('log_name')
settled_samples = pd.concat(settled_frames, ignore_index=True) if settled_frames else pd.DataFrame()
step_windows = pd.concat(step_window_frames, ignore_index=True) if step_window_frames else pd.DataFrame()

print(f'Settled positive-flow samples retained: {len(settled_samples):,}')
display(
    coverage[[
        'raw_rows', 'settled_samples', 'legacy_pre_fix_log', 'legacy_tc_correction_applied',
        'restored_tc_calibration_applied', 'legacy_room_anchor_samples', 'pump_speed_sources', 'tmi_min_C', 'tmi_max_C',
        'speed_min_rpm', 'speed_max_rpm', 'command_levels_pct'
    ]].round(3)
)


Settled positive-flow samples retained: 24,709


,raw_rows,settled_samples,legacy_pre_fix_log,legacy_tc_correction_applied,restored_tc_calibration_applied,legacy_room_anchor_samples,pump_speed_sources,tmi_min_C,tmi_max_C,speed_min_rpm,speed_max_rpm,command_levels_pct
log_name,,,,,,,,,,,,
log_20260330_161922.csv,3580,123,True,True,False,3580,"freq_hz_x30, logged_rpm",19.883,20.194,33.0,224.0,"3, 5, 6, 11"
log_20260402_150754.csv,4793,2229,True,True,False,3002,"freq_hz_x30, logged_rpm",7.610,20.638,89.0,634.0,"5, 10, 20, 25, 30"
log_20260403_115916.csv,2587,1723,True,True,False,678,"freq_hz_x30, logged_rpm",-3.499,20.482,89.0,420.0,"5, 10, 20"
log_20260417_094053.csv,10091,8308,True,True,False,686,"freq_hz_x30, logged_rpm",-57.817,20.464,87.0,1267.0,"5, 40, 60"
log_20260422_102446_filling.csv,6805,0,False,False,True,0,freq_hz_x30,NaN,NaN,NaN,NaN,
log_20260422_143345.csv,11210,4509,False,False,True,0,"freq_hz_x30, logged_rpm",-65.360,20.970,89.0,1691.0,"5, 20, 40, 60, 70, 75, 80"
log_20260424_153546.csv,9467,7817,False,False,True,0,"freq_hz_x30, logged_rpm",-99.890,21.460,418.0,1267.0,"20, 30, 40, 45, 50, 55, 60"


## Validation Checks

These checks make the notebook fail loudly if the expected data products disappear or if the legacy proxy calibration path stops being used for older logs.


In [4]:
required_columns = ['TMI_C', 'volume_flow_lmin_si', 'pump_speed_rpm']
missing_required = [column for column in required_columns if column not in settled_samples.columns]
assert not missing_required, f'Missing required columns after normalization: {missing_required}'

for column in required_columns:
    finite_count = int(np.isfinite(pd.to_numeric(settled_samples[column], errors='coerce')).sum())
    assert finite_count > 0, f'{column} has no finite values after filtering.'

legacy_coverage = coverage[coverage['legacy_pre_fix_log'].astype(bool)]
assert not legacy_coverage.empty, 'No pre-fix legacy recirculation logs were found.'
assert legacy_coverage['legacy_tc_correction_applied'].astype(bool).all(), (
    'At least one pre-fix recirculation log did not report legacy TC correction metadata.'
)

print('Required normalized columns are populated.')
print(f'Pre-fix logs with proxy TC correction: {len(legacy_coverage)}')


Required normalized columns are populated.
Pre-fix logs with proxy TC correction: 4


## Temperature and Speed Binning

Temperature bins are automatic 5 °C intervals using pump inlet temperature `TMI_C`. Speed groups are pump command buckets, but the plotted x-value is the measured mean pump speed in rpm for that group.


In [5]:
def make_temperature_edges(values: pd.Series, width_c: float) -> np.ndarray:
    finite = pd.to_numeric(values, errors='coerce').dropna()
    if finite.empty:
        raise RuntimeError('No finite inlet temperatures are available for binning.')
    lo = width_c * np.floor(float(finite.min()) / width_c)
    hi = width_c * np.ceil(float(finite.max()) / width_c)
    return np.arange(lo, hi + width_c, width_c)


def interval_label(interval: pd.Interval) -> str:
    return f'{interval.left:g} to {interval.right:g} °C'


def interval_center(interval: pd.Interval) -> float:
    return 0.5 * (float(interval.left) + float(interval.right))


samples = settled_samples.copy()
temperature_edges = make_temperature_edges(samples['TMI_C'], TEMP_BIN_WIDTH_C)
samples['tmi_bin'] = pd.cut(
    samples['TMI_C'],
    bins=temperature_edges,
    right=False,
    include_lowest=True,
)
samples['tmi_bin_label'] = samples['tmi_bin'].map(interval_label)
samples['tmi_bin_center_C'] = samples['tmi_bin'].map(interval_center).astype(float)

# Derived pump-performance quantities.
samples['delivered_ml_per_rev'] = samples['volume_flow_lmin_si'] * 1000.0 / samples['pump_speed_rpm']
samples['specific_input_energy_j_per_l'] = 60.0 * samples['pump_input_power_w'] / samples['volume_flow_lmin_si']
samples['hydraulic_power_w'] = samples['delta_p_bar_recomputed'] * samples['volume_flow_lmin_si'] * 1.0e5 * 1.0e-3 / 60.0
samples['efficiency_proxy'] = samples['hydraulic_power_w'] / samples['pump_input_power_w']

binned_all = (
    samples.dropna(subset=['tmi_bin'])
    .groupby(['tmi_bin', 'cmd_bucket_pct'], observed=True)
    .agg(
        sample_count=('volume_flow_lmin_si', 'size'),
        log_count=('log_name', 'nunique'),
        logs=('log_name', lambda series: ', '.join(sorted(set(series)))),
        mean_speed_rpm=('pump_speed_rpm', 'mean'),
        std_speed_rpm=('pump_speed_rpm', 'std'),
        mean_freq_hz=('pump_freq_hz', 'mean'),
        mean_volume_flow_lmin=('volume_flow_lmin_si', 'mean'),
        std_volume_flow_lmin=('volume_flow_lmin_si', 'std'),
        mean_inlet_temp_C=('TMI_C', 'mean'),
        std_inlet_temp_C=('TMI_C', 'std'),
        mean_delta_p_bar=('delta_p_bar_recomputed', 'mean'),
        mean_power_w=('pump_input_power_w', 'mean'),
        mean_delivered_ml_per_rev=('delivered_ml_per_rev', 'mean'),
        mean_specific_energy_j_per_l=('specific_input_energy_j_per_l', 'mean'),
        mean_hydraulic_power_w=('hydraulic_power_w', 'mean'),
        mean_efficiency_proxy=('efficiency_proxy', 'mean'),
    )
    .reset_index()
)

binned_all['std_volume_flow_lmin'] = binned_all['std_volume_flow_lmin'].fillna(0.0)
binned_all['flow_lower_lmin'] = (binned_all['mean_volume_flow_lmin'] - binned_all['std_volume_flow_lmin']).clip(lower=0.0)
binned_all['flow_upper_lmin'] = binned_all['mean_volume_flow_lmin'] + binned_all['std_volume_flow_lmin']
binned_all['tmi_bin_label'] = binned_all['tmi_bin'].map(interval_label)
binned_all['tmi_bin_center_C'] = binned_all['tmi_bin'].map(interval_center).astype(float)

speed_temp = binned_all.loc[binned_all['sample_count'].ge(MIN_SAMPLES_PER_SPEED_TEMP)].copy()
eligible_bin_counts = speed_temp.groupby('tmi_bin', observed=True)['cmd_bucket_pct'].nunique()
eligible_bins = eligible_bin_counts.loc[eligible_bin_counts.ge(MIN_SPEEDS_PER_TEMP_BIN)].index
pump_curve = speed_temp.loc[speed_temp['tmi_bin'].isin(eligible_bins)].copy()
pump_curve = pump_curve.sort_values(['tmi_bin_center_C', 'mean_speed_rpm'])

room_mask = pump_curve['tmi_bin'].map(lambda interval: interval.left <= 20.0 < interval.right).astype(bool)
cold_mask = pump_curve['tmi_bin_center_C'].lt(20.0)
assert room_mask.any(), 'No eligible room-temperature pump-inlet bin containing 20 °C was found.'
assert cold_mask.any(), 'No eligible colder pump-inlet bin with multiple speeds was found.'

print(f'Eligible temperature bands: {pump_curve["tmi_bin"].nunique()}')
print(f'Eligible speed/temperature groups: {len(pump_curve)}')
display(
    pump_curve[[
        'tmi_bin_label', 'cmd_bucket_pct', 'sample_count', 'log_count', 'mean_speed_rpm',
        'mean_volume_flow_lmin', 'std_volume_flow_lmin', 'mean_inlet_temp_C', 'logs'
    ]].round(3)
)


Eligible temperature bands: 23
Eligible speed/temperature groups: 88


,tmi_bin_label,cmd_bucket_pct,sample_count,log_count,mean_speed_rpm,mean_volume_flow_lmin,std_volume_flow_lmin,mean_inlet_temp_C,logs
0,-100 to -95 °C,40.0,1068,1,843.638,2.741,0.009,-97.572,log_20260424_153546.csv
1,-100 to -95 °C,45.0,11,1,948.818,3.085,0.003,-98.878,log_20260424_153546.csv
2,-100 to -95 °C,50.0,31,1,1054.323,3.419,0.022,-98.781,log_20260424_153546.csv
3,-100 to -95 °C,55.0,17,1,1158.941,3.746,0.003,-97.405,log_20260424_153546.csv
4,-100 to -95 °C,60.0,24,1,1265.625,4.082,0.003,-95.832,log_20260424_153546.csv
...,...,...,...,...,...,...,...,...,...
91,20 to 25 °C,11.0,20,1,224.000,0.716,0.001,20.072,log_20260330_161922.csv
92,20 to 25 °C,20.0,329,3,418.486,1.301,0.006,20.478,"log_20260402_150754.csv, log_20260403_115916.c..."
93,20 to 25 °C,25.0,57,1,526.579,1.641,0.003,20.204,log_20260402_150754.csv
94,20 to 25 °C,30.0,374,1,633.019,1.938,0.008,20.351,log_20260402_150754.csv


## Main Pump Curve

Each line is a pump-inlet temperature band. Markers show mean volume flow at the measured mean pump speed for that command bucket. Shaded bands and capped vertical bars show mean ± one sample standard deviation in volume flow.


In [6]:
fig, ax = plt.subplots(figsize=(11.5, 7.0), constrained_layout=True)

norm = mpl.colors.Normalize(
    vmin=float(pump_curve['tmi_bin_center_C'].min()),
    vmax=float(pump_curve['tmi_bin_center_C'].max()),
)
cmap = mpl.colormaps['turbo']

for interval, group in pump_curve.groupby('tmi_bin', observed=True):
    group = group.sort_values('mean_speed_rpm')
    center_c = interval_center(interval)
    color = cmap(norm(center_c))
    contains_room = interval.left <= 20.0 < interval.right
    label = f'{interval_label(interval)}' + ('  room' if contains_room else '')
    lw = 2.2 if contains_room else 1.35
    alpha = 0.95 if contains_room else 0.72
    zorder = 5 if contains_room else 3

    ax.plot(
        group['mean_speed_rpm'],
        group['mean_volume_flow_lmin'],
        marker='o',
        ms=4.5 if contains_room else 3.5,
        lw=lw,
        alpha=alpha,
        color='black' if contains_room else color,
        label=label,
        zorder=zorder,
    )
    ax.errorbar(
        group['mean_speed_rpm'].to_numpy(float),
        group['mean_volume_flow_lmin'].to_numpy(float),
        yerr=group['std_volume_flow_lmin'].to_numpy(float),
        fmt='none',
        ecolor='black' if contains_room else color,
        elinewidth=1.15 if contains_room else 0.85,
        capsize=3.2 if contains_room else 2.2,
        capthick=1.15 if contains_room else 0.85,
        alpha=0.95 if contains_room else 0.72,
        zorder=zorder + 1,
    )
    ax.fill_between(
        group['mean_speed_rpm'].to_numpy(float),
        group['flow_lower_lmin'].to_numpy(float),
        group['flow_upper_lmin'].to_numpy(float),
        color='black' if contains_room else color,
        alpha=0.24 if contains_room else 0.14,
        linewidth=0,
        zorder=zorder - 1,
    )

ax.set_xlabel('Pump speed [rpm]')
ax.set_ylabel('Volume flow [L/min]')
ax.set_title('Recirculation pump performance: volume flow vs pump speed')
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.012)
cbar.set_label('Pump inlet temperature bin center [°C]')

plt.show()


## Error-Band Scale

The plotted uncertainty is the actual sample standard deviation within each pump-speed and inlet-temperature bin. Most bins have very low flow scatter, so the bands are often thinner than the marker and line stroke on the main plot.


In [7]:
flow_error_scale = pump_curve.copy()
flow_error_scale['relative_std_pct'] = 100.0 * flow_error_scale['std_volume_flow_lmin'] / flow_error_scale['mean_volume_flow_lmin']

std_quantiles = pd.DataFrame({
    'std_volume_flow_lmin': flow_error_scale['std_volume_flow_lmin'].quantile([0.00, 0.10, 0.25, 0.50, 0.75, 0.90, 1.00]),
    'relative_std_pct': flow_error_scale['relative_std_pct'].quantile([0.00, 0.10, 0.25, 0.50, 0.75, 0.90, 1.00]),
})
std_quantiles.index.name = 'quantile'

print('Error bars are mean ± one sample standard deviation in volume flow.')
display(std_quantiles.round({'std_volume_flow_lmin': 4, 'relative_std_pct': 2}))

display(
    flow_error_scale.sort_values('std_volume_flow_lmin', ascending=False)[[
        'tmi_bin_label', 'cmd_bucket_pct', 'sample_count', 'mean_speed_rpm',
        'mean_volume_flow_lmin', 'std_volume_flow_lmin', 'relative_std_pct'
    ]]
    .head(10)
    .round({'cmd_bucket_pct': 0, 'mean_speed_rpm': 1, 'mean_volume_flow_lmin': 3, 'std_volume_flow_lmin': 4, 'relative_std_pct': 2})
)


Error bars are mean ± one sample standard deviation in volume flow.


,std_volume_flow_lmin,relative_std_pct
quantile,,
0.00,0.0012,0.05
0.10,0.0028,0.10
0.25,0.0036,0.18
0.50,0.0059,0.29
0.75,0.0104,0.42
0.90,0.0165,0.65
1.00,0.0422,67.50


,tmi_bin_label,cmd_bucket_pct,sample_count,mean_speed_rpm,mean_volume_flow_lmin,std_volume_flow_lmin,relative_std_pct
80,15 to 20 °C,3.0,32,40.6,0.063,0.0422,67.50
87,20 to 25 °C,3.0,62,40.0,0.110,0.0379,34.56
81,15 to 20 °C,5.0,29,88.3,0.282,0.0303,10.74
2,-100 to -95 °C,50.0,31,1054.3,3.419,0.0221,0.65
16,-65 to -60 °C,40.0,606,843.9,2.671,0.0182,0.68
39,-40 to -35 °C,40.0,991,844.2,2.628,0.0177,0.68
46,-30 to -25 °C,40.0,704,844.2,2.625,0.0176,0.67
57,-15 to -10 °C,70.0,67,1477.4,4.513,0.0176,0.39
43,-35 to -30 °C,40.0,971,844.2,2.622,0.0175,0.67
36,-45 to -40 °C,60.0,475,1264.4,3.939,0.0161,0.41


## Coverage Heatmap

This heatmap shows where the speed-versus-flow curve is supported by data. Counts are settled samples after the transient step cut.


In [8]:
coverage_heatmap = (
    binned_all.pivot_table(
        index='tmi_bin_label',
        columns='cmd_bucket_pct',
        values='sample_count',
        aggfunc='sum',
        observed=True,
    )
    .fillna(0)
)

# Sort by the numeric temperature interval lower edge.
label_order = (
    binned_all[['tmi_bin', 'tmi_bin_label', 'tmi_bin_center_C']]
    .drop_duplicates()
    .sort_values('tmi_bin_center_C')
    ['tmi_bin_label']
)
coverage_heatmap = coverage_heatmap.reindex(label_order)

fig, ax = plt.subplots(figsize=(11.5, 8.0), constrained_layout=True)
image = ax.imshow(coverage_heatmap.to_numpy(float), aspect='auto', origin='lower', cmap='viridis')
ax.set_xticks(np.arange(len(coverage_heatmap.columns)))
ax.set_xticklabels([f'{float(col):g}%' for col in coverage_heatmap.columns], rotation=45, ha='right')
ax.set_yticks(np.arange(len(coverage_heatmap.index)))
ax.set_yticklabels(coverage_heatmap.index)
ax.set_xlabel('Pump command bucket')
ax.set_ylabel('Pump inlet temperature bin')
ax.set_title('Settled sample coverage by pump inlet temperature and command')
cbar = fig.colorbar(image, ax=ax, pad=0.012)
cbar.set_label('Sample count')
plt.show()


## Diagnostic Plots

These plots check whether the pump behavior is consistent across temperature, pressure rise, and electrical input. They are intended as quick screens for slip, restriction changes, and power-to-flow changes.


In [9]:
plot_samples = samples.copy()
plot_samples = plot_samples.replace([np.inf, -np.inf], np.nan)
plot_samples = plot_samples.dropna(subset=['pump_speed_rpm', 'volume_flow_lmin_si', 'TMI_C'])

fig, axes = plt.subplots(2, 2, figsize=(12.5, 9.0), constrained_layout=True)

scatter = axes[0, 0].scatter(
    plot_samples['TMI_C'],
    plot_samples['delivered_ml_per_rev'],
    c=plot_samples['pump_speed_rpm'],
    s=8,
    alpha=0.35,
    cmap='plasma',
)
axes[0, 0].set_xlabel('Pump inlet temperature TMI [°C]')
axes[0, 0].set_ylabel('Delivered volume [mL/rev]')
axes[0, 0].set_title('Slip proxy: delivered volume per revolution')
fig.colorbar(scatter, ax=axes[0, 0], pad=0.012, label='Pump speed [rpm]')

scatter = axes[0, 1].scatter(
    plot_samples['volume_flow_lmin_si'],
    plot_samples['delta_p_bar_recomputed'],
    c=plot_samples['TMI_C'],
    s=8,
    alpha=0.35,
    cmap='turbo',
)
axes[0, 1].set_xlabel('Volume flow [L/min]')
axes[0, 1].set_ylabel('Pump pressure rise [bar]')
axes[0, 1].set_title('Pressure rise versus delivered flow')
fig.colorbar(scatter, ax=axes[0, 1], pad=0.012, label='Pump inlet TMI [°C]')

for cmd, group in pump_curve.groupby('cmd_bucket_pct', observed=True):
    axes[1, 0].plot(
        group['tmi_bin_center_C'],
        group['mean_delivered_ml_per_rev'],
        marker='o',
        lw=1.2,
        ms=3.5,
        label=f'{cmd:g}%',
    )
axes[1, 0].set_xlabel('Pump inlet temperature bin center [°C]')
axes[1, 0].set_ylabel('Mean delivered volume [mL/rev]')
axes[1, 0].set_title('Delivered volume per revolution by command')
axes[1, 0].legend(title='Command', fontsize=7, ncol=2)

for interval, group in pump_curve.groupby('tmi_bin', observed=True):
    group = group.sort_values('mean_speed_rpm')
    center_c = interval_center(interval)
    color = cmap(norm(center_c))
    axes[1, 1].plot(
        group['mean_speed_rpm'],
        group['mean_specific_energy_j_per_l'],
        marker='o',
        lw=1.1,
        ms=3.2,
        alpha=0.72,
        color='black' if interval.left <= 20.0 < interval.right else color,
    )
axes[1, 1].set_xlabel('Pump speed [rpm]')
axes[1, 1].set_ylabel('Specific input energy [J/L]')
axes[1, 1].set_title('Electrical input energy per liter')

plt.show()


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.7), constrained_layout=True)

scatter = axes[0].scatter(
    plot_samples['pump_speed_rpm'],
    plot_samples['hydraulic_power_w'],
    c=plot_samples['TMI_C'],
    s=8,
    alpha=0.35,
    cmap='turbo',
)
axes[0].set_xlabel('Pump speed [rpm]')
axes[0].set_ylabel('Hydraulic power from Δp·Q [W]')
axes[0].set_title('Hydraulic output proxy')
fig.colorbar(scatter, ax=axes[0], pad=0.012, label='Pump inlet TMI [°C]')

finite_eff = plot_samples['efficiency_proxy'].replace([np.inf, -np.inf], np.nan)
valid_eff = finite_eff.between(0.0, 1.0)
scatter = axes[1].scatter(
    plot_samples.loc[valid_eff, 'pump_speed_rpm'],
    finite_eff.loc[valid_eff] * 100.0,
    c=plot_samples.loc[valid_eff, 'TMI_C'],
    s=8,
    alpha=0.35,
    cmap='turbo',
)
axes[1].set_xlabel('Pump speed [rpm]')
axes[1].set_ylabel('Hydraulic/input power proxy [%]')
axes[1].set_title('Pump efficiency proxy')
fig.colorbar(scatter, ax=axes[1], pad=0.012, label='Pump inlet TMI [°C]')

plt.show()


## Flow-Speed Fit by Temperature

A simple linear fit of mean flow versus mean speed in each inlet-temperature band is useful for checking whether the flow-speed slope changes as the HFE gets colder. The intercept is not a pump model; it is a drift/slip diagnostic for this limited operating envelope.


In [11]:
fit_rows = []
for interval, group in pump_curve.groupby('tmi_bin', observed=True):
    group = group.sort_values('mean_speed_rpm')
    x = group['mean_speed_rpm'].to_numpy(float)
    y = group['mean_volume_flow_lmin'].to_numpy(float)
    if len(group) < 2 or np.allclose(x, x[0]):
        continue
    slope, intercept = np.polyfit(x, y, deg=1)
    yhat = slope * x + intercept
    ss_res = float(np.sum((y - yhat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = np.nan if ss_tot <= 0.0 else 1.0 - ss_res / ss_tot
    fit_rows.append({
        'tmi_bin': interval,
        'tmi_bin_label': interval_label(interval),
        'tmi_bin_center_C': interval_center(interval),
        'speed_group_count': len(group),
        'sample_count': int(group['sample_count'].sum()),
        'flow_lmin_per_rpm': slope,
        'flow_lmin_at_zero_rpm': intercept,
        'r2': r2,
    })

flow_speed_fit = pd.DataFrame(fit_rows).sort_values('tmi_bin_center_C')
display(flow_speed_fit.round(5))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.7), constrained_layout=True)
axes[0].plot(flow_speed_fit['tmi_bin_center_C'], flow_speed_fit['flow_lmin_per_rpm'], marker='o', lw=1.4)
axes[0].axvline(20.0, color='0.35', lw=1.0, ls='--', alpha=0.7)
axes[0].set_xlabel('Pump inlet temperature bin center [°C]')
axes[0].set_ylabel('Flow-speed slope [L/min/rpm]')
axes[0].set_title('Linear flow-speed slope vs temperature')

axes[1].plot(flow_speed_fit['tmi_bin_center_C'], flow_speed_fit['flow_lmin_at_zero_rpm'], marker='o', lw=1.4, color='C1')
axes[1].axhline(0.0, color='0.35', lw=1.0, ls='--', alpha=0.7)
axes[1].axvline(20.0, color='0.35', lw=1.0, ls='--', alpha=0.7)
axes[1].set_xlabel('Pump inlet temperature bin center [°C]')
axes[1].set_ylabel('Fit intercept [L/min]')
axes[1].set_title('Flow-speed intercept diagnostic')

plt.show()


,tmi_bin,tmi_bin_label,tmi_bin_center_C,speed_group_count,sample_count,flow_lmin_per_rpm,flow_lmin_at_zero_rpm,r2
0,"[-100.0, -95.0)",-100 to -95 °C,-97.5,5,1151,0.00317,0.07169,0.99992
1,"[-95.0, -90.0)",-95 to -90 °C,-92.5,2,630,0.00319,0.03886,1.00000
2,"[-90.0, -85.0)",-90 to -85 °C,-87.5,2,863,0.00321,0.01377,1.00000
3,"[-75.0, -70.0)",-75 to -70 °C,-72.5,2,453,0.00317,0.00742,1.00000
4,"[-70.0, -65.0)",-70 to -65 °C,-67.5,2,382,0.00317,0.00634,1.00000
5,"[-65.0, -60.0)",-65 to -60 °C,-62.5,3,700,0.00306,0.08337,0.99999
6,"[-60.0, -55.0)",-60 to -55 °C,-57.5,5,951,0.00303,0.09782,0.99982
7,"[-55.0, -50.0)",-55 to -50 °C,-52.5,6,1346,0.00304,0.08647,0.99979
8,"[-50.0, -45.0)",-50 to -45 °C,-47.5,5,1517,0.00303,0.08275,0.99952
9,"[-45.0, -40.0)",-45 to -40 °C,-42.5,4,1573,0.00302,0.07809,0.99957


## Other Pump-Performance Checks Worth Making

Beyond the plots generated above, useful checks include:

- Repeatability by log date at the same inlet-temperature and speed bins.
- Flow-meter density and mass-flow consistency against the HFE property curve.
- Pressure rise versus speed at fixed flow or fixed command to check restriction changes.
- Warm restart behavior before and after priming transients.
- Pump electrical current versus input power to catch VFD telemetry scaling or motor-loading anomalies.
- Comparison with vendor pump curves after converting to the same fluid, temperature, and pressure-rise basis.
